# Detecção de Anomalias em Transações com Python

Este notebook apresenta o projeto completo de detecção de fraudes em transações de cartão de crédito.

O foco é resolver um problema de **classe rara**: a maioria das transações é normal e apenas uma pequena parcela é fraude. Por isso, a análise não pode depender apenas da acurácia.

Métricas principais usadas:

- Recall;
- Precision;
- F1-score;
- ROC-AUC;
- Average Precision;
- Curva Precision-Recall.


## 1. Imports

Nesta etapa importamos as bibliotecas usadas no projeto.

- `pandas` e `numpy`: manipulação e criação de variáveis;
- `matplotlib`: gráficos;
- `sklearn`: modelos, métricas, divisão treino/teste e padronização;
- `joblib`: salvamento do modelo final;
- `xgboost`: modelo avançado, se estiver instalado.


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

## 2. Caminhos do projeto

Aqui definimos os diretórios usados para dados, gráficos, métricas e modelos.

Isso deixa o notebook mais organizado e evita caminhos espalhados pelo código.


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_METRICS = PROJECT_ROOT / "outputs" / "metrics"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [OUTPUT_FIGURES, OUTPUT_METRICS, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## 3. Carregamento do dataset

O arquivo esperado é `data/creditcard.csv`.

A coluna `Class` indica:

- `0`: transação normal;
- `1`: transação fraudulenta.


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## 4. Validação inicial

Antes de treinar qualquer modelo, validamos se as colunas essenciais existem.

Isso reduz o risco de rodar o projeto com uma base errada.


In [ ]:
required_columns = {"Time", "Amount", "Class"}
missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(f"Colunas ausentes: {missing_columns}")

print("Colunas essenciais encontradas com sucesso.")
print("Classes existentes:", sorted(df["Class"].unique()))

## 5. Análise do desbalanceamento

Fraudes são raras. Por isso, a primeira análise importante é entender a proporção entre transações normais e fraudulentas.


In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_percent = df["Class"].value_counts(normalize=True).sort_index()

profile = pd.DataFrame({
    "quantidade": class_counts,
    "percentual": class_percent
})

profile

## 6. Gráfico da variável alvo

Este gráfico mostra visualmente o desbalanceamento da base.

A classe `1`, fraude, é muito pequena quando comparada com a classe `0`.


In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(["Normal (0)", "Fraude (1)"], class_counts.values)
plt.title("Distribuição da variável Class")
plt.ylabel("Quantidade de transações")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "distribuicao_class.png", dpi=150)
plt.show()

## 7. Feature Engineering

Criamos novas variáveis para ajudar o modelo a interpretar melhor os padrões.

Variáveis criadas:

- `Amount_log`: reduz o impacto de valores extremos;
- `Amount_to_median_ratio`: compara a transação com a mediana;
- `High_amount_flag`: marca valores acima do percentil 99;
- `Hour`: aproxima a hora do dia;
- `Amount_scaled` e `Time_scaled`: padronizam valor e tempo.


In [ ]:
data = df.copy()

amount_median = data["Amount"].median()
amount_p99 = data["Amount"].quantile(0.99)

# Reduz a assimetria da variável Amount.
data["Amount_log"] = np.log1p(data["Amount"])

# Compara a transação com o valor mediano da base.
data["Amount_to_median_ratio"] = data["Amount"] / (amount_median + 1e-9)

# Marca transações de valor muito alto.
data["High_amount_flag"] = (data["Amount"] > amount_p99).astype(int)

# Aproxima a hora do dia usando a coluna Time.
data["Hour"] = ((data["Time"] // 3600) % 24).astype(int)

# Padroniza Amount e Time.
scaler = StandardScaler()
data[["Amount_scaled", "Time_scaled"]] = scaler.fit_transform(data[["Amount", "Time"]])

data[["Amount", "Amount_log", "Amount_scaled", "Time", "Time_scaled", "Hour", "High_amount_flag"]].head()

## 8. Separação entre variáveis explicativas e alvo

`X` contém as variáveis usadas pelo modelo.

`y` contém a resposta que queremos prever: fraude ou não fraude.

Removemos `Amount` e `Time` originais porque criamos versões transformadas dessas variáveis.


In [ ]:
X = data.drop(columns=["Class", "Amount", "Time"])
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("Proporção no treino:")
print(y_train.value_counts(normalize=True))
print("Proporção no teste:")
print(y_test.value_counts(normalize=True))

## 9. Função de avaliação

Esta função padroniza a avaliação dos modelos.

Ela calcula:

- Accuracy;
- Precision;
- Recall;
- F1-score;
- ROC-AUC;
- Average Precision;
- Matriz de confusão.


In [ ]:
def evaluate_model(name, model, X_test, y_test, threshold=0.50):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "modelo": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "average_precision": average_precision_score(y_test, y_prob),
        "matriz_confusao": confusion_matrix(y_test, y_pred).tolist(),
    }

    print(f"Modelo: {name}")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Matriz de confusão:")
    print(confusion_matrix(y_test, y_pred))

    return metrics, y_prob

## 10. Modelo 1 - Regressão Logística Baseline

Modelo simples de referência.

Ele serve como ponto inicial para comparar se técnicas de balanceamento e modelos mais avançados melhoram o resultado.


In [ ]:
logistic_baseline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=100, solver="newton-cholesky", random_state=RANDOM_STATE))
])

logistic_baseline.fit(X_train, y_train)
metrics_logistic_baseline, prob_logistic_baseline = evaluate_model(
    "logistic_baseline",
    logistic_baseline,
    X_test,
    y_test
)

## 11. Modelo 2 - Regressão Logística Balanceada

Aqui usamos `class_weight="balanced"`.

Isso aumenta o peso da classe rara durante o treinamento, ajudando o modelo a prestar mais atenção nas fraudes.


In [ ]:
logistic_balanced = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=100,
        solver="newton-cholesky",
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

logistic_balanced.fit(X_train, y_train)
metrics_logistic_balanced, prob_logistic_balanced = evaluate_model(
    "logistic_balanced",
    logistic_balanced,
    X_test,
    y_test
)

## 12. Modelo 3 - Undersampling

O undersampling balanceia apenas a base de treino.

A base de teste continua desbalanceada, pois ela precisa representar o mundo real.


In [ ]:
train_df = X_train.copy()
train_df["Class"] = y_train.values

frauds = train_df[train_df["Class"] == 1]
normals = train_df[train_df["Class"] == 0].sample(n=len(frauds), random_state=RANDOM_STATE)

balanced_train = pd.concat([frauds, normals]).sample(frac=1, random_state=RANDOM_STATE)
X_train_under = balanced_train.drop(columns="Class")
y_train_under = balanced_train["Class"]

logistic_undersampling = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=100, solver="newton-cholesky", random_state=RANDOM_STATE))
])

logistic_undersampling.fit(X_train_under, y_train_under)
metrics_logistic_undersampling, prob_logistic_undersampling = evaluate_model(
    "logistic_undersampling",
    logistic_undersampling,
    X_test,
    y_test
)

## 13. Modelo 4 - Random Forest Balanceado

Random Forest usa várias árvores de decisão.

Com `class_weight="balanced_subsample"`, o modelo considera o desbalanceamento ao construir as árvores.


In [ ]:
random_forest_balanced = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_forest_balanced.fit(X_train, y_train)
metrics_random_forest, prob_random_forest = evaluate_model(
    "random_forest_balanced",
    random_forest_balanced,
    X_test,
    y_test
)

## 14. Modelo 5 - XGBoost Balanceado

XGBoost é um modelo avançado baseado em boosting.

Usamos `scale_pos_weight` para compensar a raridade da classe fraudulenta.


In [ ]:
try:
    from xgboost import XGBClassifier

    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

    xgboost_balanced = XGBClassifier(
        n_estimators=40,
        max_depth=3,
        learning_rate=0.10,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        tree_method="hist",
        subsample=0.90,
        colsample_bytree=0.90,
        random_state=RANDOM_STATE,
        n_jobs=2,
    )

    xgboost_balanced.fit(X_train, y_train)
    metrics_xgboost, prob_xgboost = evaluate_model(
        "xgboost_balanced",
        xgboost_balanced,
        X_test,
        y_test
    )
except Exception as exc:
    print("XGBoost não executado neste ambiente:", exc)
    metrics_xgboost = None
    prob_xgboost = None

## 15. Comparação dos modelos

Agora consolidamos as métricas para comparar os modelos.

O melhor modelo operacional não é necessariamente o que tem maior recall isolado. Também precisamos controlar falsos positivos.


In [ ]:
metrics_list = [
    metrics_logistic_baseline,
    metrics_logistic_balanced,
    metrics_logistic_undersampling,
    metrics_random_forest,
]

if metrics_xgboost is not None:
    metrics_list.append(metrics_xgboost)

metrics_df = pd.DataFrame(metrics_list)
metrics_df.to_csv(OUTPUT_METRICS / "metricas_modelos.csv", index=False)

metrics_df[["modelo", "precision", "recall", "f1_score", "roc_auc", "average_precision"]].sort_values(
    by=["average_precision", "f1_score", "recall"],
    ascending=False
)

## 16. Curvas ROC e Precision-Recall

A curva ROC ajuda a avaliar separação geral das classes.

A curva Precision-Recall é especialmente importante em bases desbalanceadas, pois mostra melhor o trade-off entre capturar fraudes e controlar falsos alertas.


In [ ]:
def save_curves(name, y_test, y_prob):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Modelo aleatório")
    plt.title(f"Curva ROC - {name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Recall")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / f"curva_roc_{name}.png", dpi=150)
    plt.show()

    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f"Average Precision = {avg_precision:.4f}")
    plt.title(f"Curva Precision-Recall - {name}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES / f"curva_precision_recall_{name}.png", dpi=150)
    plt.show()

save_curves("random_forest_balanced", y_test, prob_random_forest)

## 17. Análise de threshold

O threshold padrão é 0,50, mas em fraude esse ponto pode ser ajustado.

Abaixo testamos vários thresholds para observar o impacto em Precision, Recall e F1-score.


In [ ]:
threshold_rows = []

for threshold in np.arange(0.05, 0.96, 0.05):
    y_pred = (prob_random_forest >= threshold).astype(int)
    fraudes_capturadas = int(((y_pred == 1) & (y_test.values == 1)).sum())
    alertas = int(y_pred.sum())

    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "alertas_gerados": alertas,
        "fraudes_capturadas": fraudes_capturadas,
        "falsos_positivos": alertas - fraudes_capturadas,
        "fraudes_nao_capturadas": int(((y_pred == 0) & (y_test.values == 1)).sum()),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(OUTPUT_METRICS / "threshold_analysis.csv", index=False)
threshold_df.sort_values(by=["f1_score", "recall"], ascending=False).head(10)


## 18. Comparação direta: threshold 0,30 x 0,50 x 0,55

O threshold 0,30 foi incluído para simular uma política mais agressiva de detecção.

Ele captura mais fraudes, mas gera mais falsos positivos. Essa comparação mostra que threshold não é apenas uma escolha técnica: é uma decisão de negócio.



In [ ]:
threshold_comparison = threshold_df[threshold_df["threshold"].isin([0.30, 0.50, 0.55])].copy()
threshold_comparison["cenario"] = threshold_comparison["threshold"].map({
    0.30: "Agressivo: maior captura de fraudes",
    0.50: "Padrão operacional inicial",
    0.55: "Maior F1-score observado",
})

threshold_comparison = threshold_comparison[[
    "threshold",
    "cenario",
    "precision",
    "recall",
    "f1_score",
    "alertas_gerados",
    "fraudes_capturadas",
    "falsos_positivos",
    "fraudes_nao_capturadas",
]]

threshold_comparison.to_csv(OUTPUT_METRICS / "threshold_comparison_030_050_055.csv", index=False)
threshold_comparison


### Leitura dos resultados

No `random_forest_balanced`, o threshold 0,30 capturou 123 fraudes, contra 117 no threshold 0,50.

A vantagem foi reduzir as fraudes não capturadas de 31 para 25. O custo foi aumentar os falsos positivos de 24 para 73 e elevar os alertas de 141 para 196.

Portanto, o threshold 0,30 é interessante quando a empresa prefere investigar mais casos para reduzir perda por fraude. Já o threshold 0,50 é mais equilibrado quando a capacidade de análise manual é limitada.



## 19. Importância das variáveis

Modelos baseados em árvores permitem observar quais variáveis tiveram maior influência na decisão.

Isso ajuda na explicabilidade do projeto.


In [ ]:
importances = pd.Series(random_forest_balanced.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

importances.head(15).to_csv(OUTPUT_METRICS / "importancia_variaveis_random_forest_balanced.csv", header=["importance"])

plt.figure(figsize=(10, 6))
importances.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Variáveis Mais Importantes - Random Forest")
plt.xlabel("Importância")
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / "importancia_variaveis_random_forest_balanced.png", dpi=150)
plt.show()

## 20. Salvamento do melhor modelo

O modelo escolhido foi `random_forest_balanced`, por apresentar o melhor equilíbrio operacional.

Ele é salvo em formato `.pkl` para reutilização futura.


In [ ]:
joblib.dump(random_forest_balanced, MODEL_DIR / "melhor_modelo_fraude.pkl")
print("Modelo salvo em:", MODEL_DIR / "melhor_modelo_fraude.pkl")

## 21. Conclusão

O projeto mostra que acurácia não é suficiente em detecção de fraude.

Principais aprendizados:

- a base é extremamente desbalanceada;
- modelos com recall muito alto podem gerar falsos positivos demais;
- o melhor modelo operacional foi o que equilibrou captura de fraude e eficiência dos alertas;
- a análise de threshold é essencial para transformar o modelo em uma decisão de negócio;
- curvas Precision-Recall são muito úteis para problemas de classe rara.
